## 1. Data Collection and Preprocessing

First, let's install the necessary libraries for data handling and machine learning.

In [5]:
pip install pandas numpy scikit-learn matplotlib seaborn

The CICIDS2017 dataset is quite large and typically distributed as multiple CSV files (one for each day of the capture). You will need to download these files and upload them to your Colab environment or mount your Google Drive if they are stored there.

**Instructions to download CICIDS2017:**
1. Go to the official [CICIDS2017 dataset page](https://www.unb.ca/cic/datasets/ids-2017.html).
2. Download the seven CSV files (Monday-Friday, Thursday-Morning, Thursday-Afternoon, Wednesday-Attack, etc.).
3. Upload these files to your Colab session's file system (e.g., by dragging them into the 'Files' tab on the left panel, or using `files.upload()` if you prefer to upload them dynamically).

Once uploaded, they should be accessible in the current working directory.

Alternatively, you can save them to your Google Drive and mount your drive using the following code cell if you prefer:

In [6]:
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = '/content/drive/MyDrive/CICIDS2017_Dataset/' # Adjust this path to your dataset location

Now, let's import the necessary libraries and load the dataset. We'll combine all daily CSVs into a single DataFrame. Ensure all files are in the same directory (e.g., `/content/` or a specified `DATA_PATH`).

In [7]:
import pandas as pd
import numpy as np
import os

# Define the path where your CICIDS2017 CSV files are located
# If you uploaded them directly to Colab, this might be './'
# If using Google Drive, uncomment and use the DATA_PATH from above
DATA_PATH = './'

# List of file names for CICIDS2017 (adjust if your filenames differ slightly)
file_names = [
    'Monday-WorkingHours.pcap_ISCX.csv',
    'Tuesday-WorkingHours.pcap_ISCX.csv',
    'Wednesday-WorkingHours.pcap_ISCX.csv',
    'Thursday-WorkingHours-MorningAttack.pcap_ISCX.csv',
    'Thursday-WorkingHours-AfternoonAttack.pcap_ISCX.csv',
    'Friday-WorkingHours-Morning.pcap_ISCX.csv',
    'Friday-WorkingHours-AfternoonandExternalAttack.pcap_ISCX.csv'
]

# Load and concatenate all CSV files
df_list = []
for fname in file_names:
    fpath = os.path.join(DATA_PATH, fname)
    if os.path.exists(fpath):
        print(f"Loading {fname}...")
        # Some columns might have inconsistent names or extra spaces
        temp_df = pd.read_csv(fpath)
        temp_df.columns = temp_df.columns.str.strip().str.replace(' ', '_').str.replace('/', '_') # Clean column names
        df_list.append(temp_df)
    else:
        print(f"File not found: {fpath}. Please ensure all CICIDS2017 CSVs are in the specified DATA_PATH.")

if df_list:
    df = pd.concat(df_list, ignore_index=True)
    print("All files loaded and concatenated successfully!")
    print(f"Combined DataFrame shape: {df.shape}")
    display(df.head())
else:
    print("No data loaded. Please check the file paths and ensure files are uploaded.")

Loading Monday-WorkingHours.pcap_ISCX.csv...
Loading Tuesday-WorkingHours.pcap_ISCX.csv...
File not found: ./Wednesday-WorkingHours.pcap_ISCX.csv. Please ensure all CICIDS2017 CSVs are in the specified DATA_PATH.
File not found: ./Thursday-WorkingHours-MorningAttack.pcap_ISCX.csv. Please ensure all CICIDS2017 CSVs are in the specified DATA_PATH.
File not found: ./Thursday-WorkingHours-AfternoonAttack.pcap_ISCX.csv. Please ensure all CICIDS2017 CSVs are in the specified DATA_PATH.
Loading Friday-WorkingHours-Morning.pcap_ISCX.csv...
File not found: ./Friday-WorkingHours-AfternoonandExternalAttack.pcap_ISCX.csv. Please ensure all CICIDS2017 CSVs are in the specified DATA_PATH.
All files loaded and concatenated successfully!
Combined DataFrame shape: (1166860, 79)


,Destination_Port,Flow_Duration,Total_Fwd_Packets,Total_Backward_Packets,Total_Length_of_Fwd_Packets,Total_Length_of_Bwd_Packets,Fwd_Packet_Length_Max,Fwd_Packet_Length_Min,Fwd_Packet_Length_Mean,Fwd_Packet_Length_Std,...,min_seg_size_forward,Active_Mean,Active_Std,Active_Max,Active_Min,Idle_Mean,Idle_Std,Idle_Max,Idle_Min,Label
0,49188,4,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,49486,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


Now that we've loaded the data, let's perform some initial data cleaning as per your proposal:
*   Handling missing and infinite values
*   Feature scaling (normalization/standardization) - *This will be done after initial cleaning and feature selection.*
*   Feature selection to remove irrelevant attributes - *This will be done in subsequent steps.*
*   Encoding labels for classification tasks - *This will be done later for the target variable.*


In [12]:
if 'df' in globals():
    print("Initial DataFrame Info:")
    df.info()

    # 1. Handle missing values
    # Replace infinite values with NaN, then drop NaNs
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    initial_rows = df.shape[0]
    df.dropna(inplace=True)
    removed_rows = initial_rows - df.shape[0]
    print(f"\nRemoved {removed_rows} rows with NaN values.")
    print(f"DataFrame shape after dropping NaNs: {df.shape}")

    # Convert all column names to string type to avoid potential errors
    df.columns = df.columns.astype(str)

    # Display data types and check for any remaining issues
    print("\nDataFrame Info after cleaning NaNs:")
    df.info()

    # Check for duplicate rows
    duplicate_rows = df.duplicated().sum()
    if duplicate_rows > 0:
        print(f"\nFound {duplicate_rows} duplicate rows. Removing duplicates...")
        df.drop_duplicates(inplace=True)
        print(f"DataFrame shape after removing duplicates: {df.shape}")
    else:
        print("\nNo duplicate rows found.")
else:
    print("DataFrame 'df' not found. Please ensure the data loading cell (cell 3d848c2b) was executed successfully after uploading the dataset files.")


Initial DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
Index: 1103868 entries, 0 to 1166859
Data columns (total 69 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   Destination_Port             1103868 non-null  float64
 1   Flow_Duration                1103868 non-null  float64
 2   Total_Fwd_Packets            1103868 non-null  float64
 3   Total_Backward_Packets       1103868 non-null  float64
 4   Total_Length_of_Fwd_Packets  1103868 non-null  float64
 5   Total_Length_of_Bwd_Packets  1103868 non-null  float64
 6   Fwd_Packet_Length_Max        1103868 non-null  float64
 7   Fwd_Packet_Length_Min        1103868 non-null  float64
 8   Fwd_Packet_Length_Mean       1103868 non-null  float64
 9   Fwd_Packet_Length_Std        1103868 non-null  float64
 10  Bwd_Packet_Length_Max        1103868 non-null  float64
 11  Bwd_Packet_Length_Min        1103868 non-null  float64
 12  Bwd_Packet_Length_Mean 

### 2. Feature Scaling, Selection, and Label Encoding

In [13]:
from sklearn.preprocessing import StandardScaler

if 'df' in globals():
    print("\n--- Feature Scaling ---")
    # Separate features and labels before scaling
    X = df.drop('Label', axis=1)
    y = df['Label']

    # Identify numerical columns for scaling
    numerical_cols = X.select_dtypes(include=np.number).columns

    if not numerical_cols.empty:
        scaler = StandardScaler()
        X_scaled = X.copy()
        X_scaled[numerical_cols] = scaler.fit_transform(X[numerical_cols])
        print(f"Scaled {len(numerical_cols)} numerical features.")
        # Recombine with label for subsequent steps
        df_scaled = pd.concat([X_scaled, y], axis=1)
        print("Features scaled successfully. Displaying head of scaled data:")
        display(df_scaled.head())
    else:
        print("No numerical columns found to scale.")

    # Update df to df_scaled for subsequent operations
    df = df_scaled.copy()

else:
    print("DataFrame 'df' not found. Please load the data first.")


--- Feature Scaling ---
Scaled 68 numerical features.
Features scaled successfully. Displaying head of scaled data:


,Destination_Port,Flow_Duration,Total_Fwd_Packets,Total_Backward_Packets,Total_Length_of_Fwd_Packets,Total_Length_of_Bwd_Packets,Fwd_Packet_Length_Max,Fwd_Packet_Length_Min,Fwd_Packet_Length_Mean,Fwd_Packet_Length_Std,...,min_seg_size_forward,Active_Mean,Active_Std,Active_Max,Active_Min,Idle_Mean,Idle_Std,Idle_Max,Idle_Min,Label
0,1.925295,-0.376896,-0.010596,-0.011030,-0.085537,-0.007836,-0.377794,-0.385250,-0.425218,-0.369223,...,0.004082,-0.114745,-0.118655,-0.149581,-0.084957,-0.275609,-0.087069,-0.278064,-0.266202,0
1,1.925295,-0.376897,-0.010596,-0.011030,-0.085537,-0.007836,-0.377794,-0.385250,-0.425218,-0.369223,...,0.004082,-0.114745,-0.118655,-0.149581,-0.084957,-0.275609,-0.087069,-0.278064,-0.266202,0
4,1.939872,-0.376896,-0.010596,-0.011030,-0.085537,-0.007836,-0.377794,-0.385250,-0.425218,-0.369223,...,0.004082,-0.114745,-0.118655,-0.149581,-0.084957,-0.275609,-0.087069,-0.278064,-0.266202,0
5,1.939872,-0.376897,-0.010596,-0.011030,-0.085537,-0.007836,-0.377794,-0.385250,-0.425218,-0.369223,...,0.004082,-0.114745,-0.118655,-0.149581,-0.084957,-0.275609,-0.087069,-0.278064,-0.266202,0
8,-0.476424,-0.376876,-0.005310,-0.007863,-0.013129,-0.007690,0.086926,-0.540336,0.163311,0.364234,...,0.004082,-0.114745,-0.118655,-0.149581,-0.084957,-0.275609,-0.087069,-0.278064,-0.266202,0


In [14]:
if 'df' in globals():
    print("\n--- Feature Selection ---")
    initial_features = df.shape[1]

    # Remove columns with a single unique value (constant features)
    constant_features = [col for col in df.columns if df[col].nunique() == 1]
    if constant_features:
        df.drop(columns=constant_features, inplace=True)
        print(f"Removed {len(constant_features)} constant features: {constant_features}")
    else:
        print("No constant features found.")

    # Optionally, remove features with very high cardinality that are not labels/identifiers
    # This is a heuristic and might need adjustment based on EDA
    high_cardinality_features = []
    for col in df.select_dtypes(include=['object']).columns:
        if col != 'Label' and df[col].nunique() > 100: # Threshold can be adjusted
            high_cardinality_features.append(col)

    if high_cardinality_features:
        df.drop(columns=high_cardinality_features, inplace=True)
        print(f"Removed {len(high_cardinality_features)} high cardinality features (excluding Label): {high_cardinality_features}")
    else:
        print("No high cardinality features (excluding Label) found for removal.")

    print(f"Features after selection: {df.shape[1]} (Removed {initial_features - df.shape[1]} features).")
    print("DataFrame info after feature selection:")
    df.info()
else:
    print("DataFrame 'df' not found. Please load and scale the data first.")


--- Feature Selection ---
No constant features found.
No high cardinality features (excluding Label) found for removal.
Features after selection: 69 (Removed 0 features).
DataFrame info after feature selection:
<class 'pandas.core.frame.DataFrame'>
Index: 1103868 entries, 0 to 1166859
Data columns (total 69 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   Destination_Port             1103868 non-null  float64
 1   Flow_Duration                1103868 non-null  float64
 2   Total_Fwd_Packets            1103868 non-null  float64
 3   Total_Backward_Packets       1103868 non-null  float64
 4   Total_Length_of_Fwd_Packets  1103868 non-null  float64
 5   Total_Length_of_Bwd_Packets  1103868 non-null  float64
 6   Fwd_Packet_Length_Max        1103868 non-null  float64
 7   Fwd_Packet_Length_Min        1103868 non-null  float64
 8   Fwd_Packet_Length_Mean       1103868 non-null  float64
 9   Fwd_Packet_Leng

### 3.5 Generative AI for Incident Reporting

To integrate a generative AI component, we would typically use a Large Language Model (LLM) via an API (e.g., Google's Gemini, OpenAI's GPT, etc.). For this project's scope, we'll demonstrate the concept by creating a function that generates human-readable incident reports based on the classification results. This function will simulate what an LLM could achieve by providing context-aware descriptions for detected threats.

In [22]:
if 'model_results' in globals() and 'y_pred_xgb' in globals() and 'y_test' in globals():
    print("\n--- Automated Incident Report Generation ---")

    # Mapping for encoded labels back to descriptive attack types (example mapping)
    # This mapping should ideally come from the LabelEncoder's classes if they were descriptive strings.
    # Assuming 0 is Benign and others are various attacks.
    # Based on value counts in preprocessing, we have 4 classes (0, 1, 2, 3)
    attack_type_mapping = {
        0: "Benign Traffic",
        1: "Port Scan Attack",
        2: "DDoS Attack",
        3: "Brute Force Attack"
    }

    def generate_incident_report(predicted_label):
        attack_description = attack_type_mapping.get(predicted_label, "Unknown/Undetermined Attack Type")

        if predicted_label == 0:
            return f"Traffic classified as {attack_description}. No malicious activity detected."
        else:
            report = f"Suspicious activity detected!\n"
            report += f"Classification: {attack_description}.\n"
            report += f"Characteristics: \n"
            if predicted_label == 1: # Port Scan
                report += "  - Numerous connection attempts to various ports."
                report += "  - Indicates reconnaissance phase by an attacker seeking open ports."
            elif predicted_label == 2: # DDoS
                report += "  - High volume of traffic from multiple sources targeting a single destination.\n"
                report += "  - Objective is to overwhelm system resources and deny service."
            elif predicted_label == 3: # Brute Force
                report += "  - Repeated, failed authentication attempts to a service.\n"
                report += "  - Suggests an attacker is trying to guess credentials."
            else:
                report += "  - Specific characteristics require further investigation."
            report += "\nRecommendation: Investigate source IPs, block suspicious traffic, and review system logs."
            return report

    # Demonstrate report generation for a few predicted attack instances from X_test
    # Find indices where XGBoost predicted an attack (labels 1, 2, or 3)
    attack_indices = [i for i, label in enumerate(y_pred_xgb) if label != 0]

    if attack_indices:
        print("\n--- Sample Incident Reports (based on XGBoost predictions) ---")
        for i in range(min(3, len(attack_indices))): # Generate 3 sample reports
            idx_in_test_set = attack_indices[i]
            predicted_attack_type = y_pred_xgb[idx_in_test_set]
            true_label = y_test.iloc[idx_in_test_set] # Get original label from y_test

            print(f"\nReport for instance (index {idx_in_test_set}):")
            print(f"  True Label: {attack_type_mapping.get(true_label, 'Unknown')}")
            print(f"  Predicted Label: {attack_type_mapping.get(predicted_attack_type, 'Unknown')}")
            print(generate_incident_report(predicted_attack_type))
    else:
        print("No attack instances predicted by XGBoost in the test set to generate reports for.")

    # Also show a benign report example
    benign_indices = [i for i, label in enumerate(y_pred_xgb) if label == 0]
    if benign_indices:
        print("\n--- Sample Benign Report ---")
        idx_in_test_set = benign_indices[0]
        predicted_benign_type = y_pred_xgb[idx_in_test_set]
        print(f"\nReport for instance (index {idx_in_test_set}):")
        print(generate_incident_report(predicted_benign_type))

else:
    print("Model results or predictions not found. Please ensure models were trained successfully.")


--- Automated Incident Report Generation ---

--- Sample Incident Reports (based on XGBoost predictions) ---

Report for instance (index 140):
  True Label: Port Scan Attack
  Predicted Label: Port Scan Attack
Suspicious activity detected!
Classification: Port Scan Attack.
Characteristics: 
  - Numerous connection attempts to various ports.  - Indicates reconnaissance phase by an attacker seeking open ports.
Recommendation: Investigate source IPs, block suspicious traffic, and review system logs.

Report for instance (index 238):
  True Label: DDoS Attack
  Predicted Label: DDoS Attack
Suspicious activity detected!
Classification: DDoS Attack.
Characteristics: 
  - High volume of traffic from multiple sources targeting a single destination.
  - Objective is to overwhelm system resources and deny service.
Recommendation: Investigate source IPs, block suspicious traffic, and review system logs.

Report for instance (index 259):
  True Label: Port Scan Attack
  Predicted Label: Port Scan

In [15]:
from sklearn.preprocessing import LabelEncoder

if 'df' in globals() and 'Label' in df.columns:
    print("\n--- Label Encoding ---")
    le = LabelEncoder()
    df['Label'] = le.fit_transform(df['Label'])
    print("Label column encoded successfully.")
    print("Original labels and their encoded values:")
    for i, label in enumerate(le.classes_):
        print(f"  {label}: {i}")
    print(f"DataFrame head after label encoding:\n{df.head()}")
    print("Value counts for encoded 'Label' column:")
    print(df['Label'].value_counts())
else:
    if 'df' not in globals():
        print("DataFrame 'df' not found. Please load, scale, and select features first.")
    else:
        print("Label column not found in DataFrame. Please check previous steps.")


--- Label Encoding ---
Label column encoded successfully.
Original labels and their encoded values:
  0: 0
  1: 1
  2: 2
  3: 3
DataFrame head after label encoding:
   Destination_Port  Flow_Duration  Total_Fwd_Packets  Total_Backward_Packets  \
0          1.925295      -0.376896          -0.010596               -0.011030   
1          1.925295      -0.376897          -0.010596               -0.011030   
4          1.939872      -0.376896          -0.010596               -0.011030   
5          1.939872      -0.376897          -0.010596               -0.011030   
8         -0.476424      -0.376876          -0.005310               -0.007863   

   Total_Length_of_Fwd_Packets  Total_Length_of_Bwd_Packets  \
0                    -0.085537                    -0.007836   
1                    -0.085537                    -0.007836   
4                    -0.085537                    -0.007836   
5                    -0.085537                    -0.007836   
8                    -0.013129  

## 3. Model Development and Training

First, we need to split our data into training and testing sets. This is crucial for evaluating model performance on unseen data and preventing overfitting.

In [16]:
from sklearn.model_selection import train_test_split

if 'df' in globals() and 'Label' in df.columns:
    print("\n--- Data Splitting ---")
    X = df.drop('Label', axis=1)
    y = df['Label']

    # Stratified split to ensure classes are well represented in both train and test sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

    print(f"Original dataset shape: {df.shape}")
    print(f"X_train shape: {X_train.shape}")
    print(f"X_test shape: {X_test.shape}")
    print(f"y_train shape: {y_train.shape}")
    print(f"y_test shape: {y_test.shape}")

    print("Data split successfully.")
else:
    print("DataFrame 'df' or 'Label' column not found. Please ensure previous preprocessing steps were completed.")


--- Data Splitting ---
Original dataset shape: (1103868, 69)
X_train shape: (772707, 68)
X_test shape: (331161, 68)
y_train shape: (772707,)
y_test shape: (331161,)
Data split successfully.


### 3.1 Supervised Learning Models

#### Random Forest Classifier (Baseline Model)

In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score
import time

if 'X_train' in globals() and 'y_train' in globals():
    print("\n--- Random Forest Classifier ---")
    # Initialize and train the Random Forest Classifier
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1) # n_jobs=-1 uses all available processors

    start_time = time.time()
    rf_model.fit(X_train, y_train)
    end_time = time.time()
    print(f"Random Forest training time: {end_time - start_time:.2f} seconds")

    # Make predictions
    y_pred_rf = rf_model.predict(X_test)

    # Evaluate the model
    print("\nRandom Forest Classification Report:")
    print(classification_report(y_test, y_pred_rf))
    print(f"Random Forest Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
    print(f"Random Forest F1-Score (macro avg): {f1_score(y_test, y_pred_rf, average='macro'):.4f}")

    # Store results for comparison
    model_results = {
        'Random Forest': {
            'Accuracy': accuracy_score(y_test, y_pred_rf),
            'Precision': precision_score(y_test, y_pred_rf, average='macro'),
            'Recall': recall_score(y_test, y_pred_rf, average='macro'),
            'F1-Score': f1_score(y_test, y_pred_rf, average='macro'),
            'Training Time': end_time - start_time
        }
    }
else:
    print("Training data (X_train, y_train) not found. Please ensure data splitting was completed.")


--- Random Forest Classifier ---
Random Forest training time: 168.77 seconds

Random Forest Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    327832
           1       0.99      0.94      0.96       584
           2       1.00      1.00      1.00      1779
           3       1.00      0.99      1.00       966

    accuracy                           1.00    331161
   macro avg       1.00      0.98      0.99    331161
weighted avg       1.00      1.00      1.00    331161

Random Forest Accuracy: 0.9999
Random Forest F1-Score (macro avg): 0.9905


#### Gradient Boosting Classifier (XGBoost)

In [18]:
import xgboost as xgb

if 'X_train' in globals() and 'y_train' in globals():
    print("\n--- XGBoost Classifier ---")
    # Initialize and train the XGBoost Classifier
    # Using objective='multi:softmax' for multi-class classification and num_class for the number of unique classes
    xgb_model = xgb.XGBClassifier(objective='multi:softmax', num_class=len(y_train.unique()),
                                  n_estimators=100, random_state=42, n_jobs=-1, eval_metric='mlogloss')

    start_time = time.time()
    xgb_model.fit(X_train, y_train)
    end_time = time.time()
    print(f"XGBoost training time: {end_time - start_time:.2f} seconds")

    # Make predictions
    y_pred_xgb = xgb_model.predict(X_test)

    # Evaluate the model
    print("\nXGBoost Classification Report:")
    print(classification_report(y_test, y_pred_xgb))
    print(f"XGBoost Accuracy: {accuracy_score(y_test, y_pred_xgb):.4f}")
    print(f"XGBoost F1-Score (macro avg): {f1_score(y_test, y_pred_xgb, average='macro'):.4f}")

    # Store results
    model_results['XGBoost'] = {
        'Accuracy': accuracy_score(y_test, y_pred_xgb),
        'Precision': precision_score(y_test, y_pred_xgb, average='macro'),
        'Recall': recall_score(y_test, y_pred_xgb, average='macro'),
        'F1-Score': f1_score(y_test, y_pred_xgb, average='macro'),
        'Training Time': end_time - start_time
    }
else:
    print("Training data (X_train, y_train) not found. Please ensure data splitting was completed.")


--- XGBoost Classifier ---
XGBoost training time: 54.25 seconds

XGBoost Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    327832
           1       0.98      0.99      0.98       584
           2       1.00      1.00      1.00      1779
           3       1.00      1.00      1.00       966

    accuracy                           1.00    331161
   macro avg       0.99      1.00      1.00    331161
weighted avg       1.00      1.00      1.00    331161

XGBoost Accuracy: 0.9999
XGBoost F1-Score (macro avg): 0.9955


### 3.2 Unsupervised Learning (Anomaly Detection)

#### Isolation Forest

In [19]:
from sklearn.ensemble import IsolationForest

if 'X_train' in globals() and 'X_test' in globals():
    print("\n--- Isolation Forest (Anomaly Detection) ---")
    # Isolation Forest is typically trained on 'normal' data to learn its characteristics.
    # For demonstration, we'll train on the entire training set and predict anomalies.
    # Adjust contamination based on expected anomaly ratio (e.g., 0.01 for 1% anomalies).
    # A low contamination value is often appropriate for network anomaly detection.
    iso_forest = IsolationForest(random_state=42, n_jobs=-1, contamination=0.01)

    start_time = time.time()
    iso_forest.fit(X_train)
    end_time = time.time()
    print(f"Isolation Forest training time: {end_time - start_time:.2f} seconds")

    # Predict anomaly scores (lower score indicates higher anomaly)
    # -1 for outliers, 1 for inliers
    y_pred_iso_forest_train = iso_forest.predict(X_train)
    y_pred_iso_forest_test = iso_forest.predict(X_test)

    print(f"\nAnomaly detection results on test set (first 20 predictions):\n{y_pred_iso_forest_test[:20]}")
    print(f"Number of anomalies detected in test set (-1): {np.sum(y_pred_iso_forest_test == -1)}")
    print(f"Number of normal instances detected in test set (1): {np.sum(y_pred_iso_forest_test == 1)}")

    # Note: Evaluating Isolation Forest performance requires careful definition of 'true' anomalies,
    # which might not directly align with the 'Label' column if it contains specific attack types.
    # This is a general anomaly detector.
else:
    print("Training data (X_train, X_test) not found. Please ensure data splitting was completed.")


--- Isolation Forest (Anomaly Detection) ---
Isolation Forest training time: 4.54 seconds

Anomaly detection results on test set (first 20 predictions):
[ 1  1  1  1  1  1 -1  1  1  1  1  1  1  1  1  1  1  1  1  1]
Number of anomalies detected in test set (-1): 3179
Number of normal instances detected in test set (1): 327982


### 3.3 Neural Network Model

#### Feedforward Neural Network (MLP)

In [20]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical

if 'X_train' in globals() and 'y_train' in globals():
    print("\n--- Feedforward Neural Network ---")
    # Convert labels to one-hot encoding for multi-class classification with Keras
    num_classes = len(y_train.unique())
    y_train_encoded = to_categorical(y_train, num_classes=num_classes)
    y_test_encoded = to_categorical(y_test, num_classes=num_classes)

    # Define the model architecture
    model = Sequential([
        Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(num_classes, activation='softmax') # Output layer with softmax for multi-class classification
    ])

    # Compile the model
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

    model.summary()

    # Train the model
    start_time = time.time()
    history = model.fit(X_train, y_train_encoded, epochs=10, batch_size=32, validation_split=0.2, verbose=1)
    end_time = time.time()
    print(f"Neural Network training time: {end_time - start_time:.2f} seconds")

    # Evaluate the model
    loss, accuracy = model.evaluate(X_test, y_test_encoded, verbose=0)
    print(f"\nNeural Network Test Accuracy: {accuracy:.4f}")

    # Predict and calculate other metrics
    y_pred_nn_prob = model.predict(X_test)
    y_pred_nn = np.argmax(y_pred_nn_prob, axis=1)

    print("\nNeural Network Classification Report:")
    print(classification_report(y_test, y_pred_nn))
    print(f"Neural Network F1-Score (macro avg): {f1_score(y_test, y_pred_nn, average='macro'):.4f}")

    # Store results
    model_results['Neural Network'] = {
        'Accuracy': accuracy,
        'Precision': precision_score(y_test, y_pred_nn, average='macro'),
        'Recall': recall_score(y_test, y_pred_nn, average='macro'),
        'F1-Score': f1_score(y_test, y_pred_nn, average='macro'),
        'Training Time': end_time - start_time
    }

    # Optional: Plot training history
    # import matplotlib.pyplot as plt
    # plt.figure(figsize=(12, 4))
    # plt.subplot(1, 2, 1)
    # plt.plot(history.history['accuracy'], label='Training Accuracy')
    # plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    # plt.title('Model Accuracy')
    # plt.xlabel('Epoch')
    # plt.ylabel('Accuracy')
    # plt.legend()
    # plt.subplot(1, 2, 2)
    # plt.plot(history.history['loss'], label='Training Loss')
    # plt.plot(history.history['val_loss'], label='Validation Loss')
    # plt.title('Model Loss')
    # plt.xlabel('Epoch')
    # plt.ylabel('Loss')
    # plt.legend()
    # plt.show()

else:
    print("Training data (X_train, y_train) not found. Please ensure data splitting was completed.")


--- Feedforward Neural Network ---


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         8,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,348 (67.77 KB)

 Trainable params: 17,348 (67.77 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
19318/19318 ━━━━━━━━━━━━━━━━━━━━ 41s 2ms/step - accuracy: 0.9968 - loss: 0.0157 - val_accuracy: 0.9989 - val_loss: 0.0074
Epoch 2/10
19318/19318 ━━━━━━━━━━━━━━━━━━━━ 38s 2ms/step - accuracy: 0.9981 - loss: 0.0103 - val_accuracy: 0.9986 - val_loss: 0.0076
Epoch 3/10
19318/19318 ━━━━━━━━━━━━━━━━━━━━ 38s 2ms/step - accuracy: 0.9982 - loss: 0.0116 - val_accuracy: 0.9988 - val_loss: 0.0065
Epoch 4/10
19318/19318 ━━━━━━━━━━━━━━━━━━━━ 39s 2ms/step - accuracy: 0.9984 - loss: 0.0084 - val_accuracy: 0.9988 - val_loss: 0.0068
Epoch 5/10
19318/19318 ━━━━━━━━━━━━━━━━━━━━ 38s 2ms/step - accuracy: 0.9984 - loss: 0.0083 - val_accuracy: 0.9972 - val_loss: 0.0111
Epoch 6/10
19318/19318 ━━━━━━━━━━━━━━━━━━━━ 41s 2ms/step - accuracy: 0.9984 - loss: 0.0084 - val_accuracy: 0.9987 - val_loss: 0.0064
Epoch 7/10
19318/19318 ━━━━━━━━━━━━━━━━━━━━ 38s 2ms/step - accuracy: 0.9985 - loss: 0.0154 - val_accuracy: 0.9972 - val_loss: 0.0072
Epoch 8/10
19318/19318 ━━━━━━━━━━━━━━━━━━━━ 37s 2ms/step - accuracy: 

### 3.4 Model Evaluation Summary

In [21]:
import pandas as pd

if 'model_results' in globals():
    print("\n--- Model Performance Summary ---")
    results_df = pd.DataFrame.from_dict(model_results, orient='index')
    display(results_df.round(4))
else:
    print("No model results found to summarize. Please ensure models were trained successfully.")


--- Model Performance Summary ---


,Accuracy,Precision,Recall,F1-Score,Training Time
Random Forest,0.9999,0.9968,0.9844,0.9905,168.7666
XGBoost,0.9999,0.9948,0.9961,0.9955,54.2482
Neural Network,0.9989,0.9952,0.8728,0.9214,385.0276
